# LangChain & LangGraph

`작성자: 최원칠 강사`   
`traveler4d@gmail.com`   

LangChain은 LLM 모델에 입력 프롬프트를 다양한 엔지니어링 기법을 추가하여 하나의 애플리케이션으로 구성할 수 있게 단계별 워크플로우를 제공합니다.

여기에 LangGraph를 추가로 활용하여 단순 직선적인 워크플로우가 아닌 원하는 도구사용과 순환, 통제방식 등을 적용한 고도화된 플로우인 에이전트를 만들어낼 수 있습니다.

## LangChain이란?

 파이썬 등에서 **LLM(대규모 언어 모델)** 기반 애플리케이션을 쉽게 구성하기 위해 만들어진 **오픈소스 라이브러리**입니다. 주로 LLM을 활용한 업무 로직과 기능(도구, 검색, 메모리, 벡터 DB 연결 등)을 쉽게 결합하여 완성된 에이전트나 애플리케이션을 구현할 수 있도록 돕는 역할을 합니다.

[LangChain 문서](https://python.langchain.com/docs/introduction/)

### 주요 구성 요소

- **채팅 모델** : LLM API를 통해 노출되는 LLM으로, 메시지 시퀀스를 입력으로 처리하고 메시지를 출력

- **Message**: 모델과 주고받는 대화의 기본 단위로 역할(Role)에 따라 메시지를 구조화하여 전달

- **Tool**: 에이전트 구성에 필요한 도구를 만들거나 가져오기 위한 인터페이스


- **메모리**: 대화나 이전 단계 결과를 기억하여, 후속 응답 시 반영할 수 있게 지원

- **벡터저장소**: 벡터 임베딩을 저장해두고, 유사 검색(문서 검색)을 수행하기 위한 인터페이스 (Faiss, Pinecone 등)

### LangChain의 장점

- **추상화된 구조**: 프롬프트, 체인, 에이전트, 메모리, 검색 등 LLM 활용에 필요한 요소들을 일관성 있게 제공

- **개발 편의성**: 코드 몇 줄로 벡터 DB 연동, Tool 사용 등을 간단히 설정할 수 있음

- **확장성**: 다양한 모델(OpenAI, Hugging Face 등)과 Vector DB(Pinecone, Faiss, Chroma 등)를 플러그인 형태로 연동 가능

### LangChain의 단계별 워크플로우

    1. 원하는 제공자의 LLM 모델을 로드
    2. 필요한 기본 메시지 형태를 작성 (시스템 메시지)
    3. 에이전트에 필요한 도구를 작성 또는 기존 도구를 로드
    4. 대화 기억을 위한 장기 또는 단기 메모리를 설정
    5. LLM 모델, 시스템 메시지, 도구, 메모리를 연결하여 하나의 애플리케이션을 구축

### LangChain 설치

LangChain을 사용하기 위해선 `langchain`라이브러리 뿐만 아니라 사용하고자 하는 LLM 모델에 제공자와 맞는 추가 패키지(`langchain-[provider]`)를 설치해야 합니다.

[모델 제공자에 따른 라이브러리](https://docs.langchain.com/oss/python/integrations/providers/overview)

추가로 메모리 사용을 위해 우선적으로 `langgraph` 패키지도 같이 설치해 줍니다.

In [ ]:
!pip install langchain langchain_community langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 22.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.8 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [langchain_community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## LangChain에 채팅 모델 불러오기
LangChain에서 LLM 모델을 사용하기 위해서는 LangChain이 지원하는 다양한 LLM 모델중 원하는 모델을 택하여 불러와야 합니다.
이때 라이센스가 필요한 모델은 해당 모델을 제공하는 도메인 사이트에서 사용 권한을 먼저 등록해야 합니다.

[지원 모델 참고](https://docs.langchain.com/oss/python/integrations/chat)

- **Tool calling** : 툴사용 가능여부 (에이전트 학습 여부)
- **Structured output** : Json과 같은 구조화된 출력 가능 여부
- **Multimodal** : 텍스트 뿐만 아닌 이미지,음성 등 여러 형식의 입출력 지원 여부

### API 모델 로드

채팅 모델을 랭체인으로 가져오기 위해 먼저 원하는 제공자 사이트에서 모델 사용허가를 진행 해주어야 합니다. 주로 API 키를 제공 받는 형식으로 사용허가를 받을 수 있습니다.

**OpenAI API 활성화**

- [OpenAI](https://platform.openai.com/docs/overview)  사이트에서 회원 가입 및 로그인 절차 진행
- 계정 설정의 [Billing](https://platform.openai.com/settings/organization/billing/overview) 페이지에서 결제 수단 등록
- Dashboard 메뉴의  [API Keys](https://platform.openai.com/api-keys) 페이지에서 API-Key 생성
- 생성된 API-Key는 복사하여 보안에 주의하여 보관

**제미나이 API 받기**
- [제미나이 API](https://aistudio.google.com/app/apikey?hl=ko) 접속
- 필수 약관 동의
- 대시보드에서 `API 키 만들기` 클릭
- `프로젝트 검색` 또는 `새 프로젝에서 만들기`
- 생성된 API 키를 복사하여 저장


API 키를 제공 받았다면 이제 키 정보를 사용하여 LLM 모델을 LangChain으로 불러옵니다

**채팅 모델 불러오기**


- 환경 변수에 API 키 저장 (`os.environ["GOOGLE_API_KEY"]`)

- `langchain.chat_models` 모듈에서 `init_chat_model()` 함수에 원하는 채팅 모델의 이름과 제공처를 입력

- `invoke()` 함수를 통해 간단하게 프롬프트를 입력하여 사용 가능


In [ ]:
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.9 MB/s  0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain-google-genai]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from langchain.chat_models import init_chat_model
import os
import getpass


# 환경 변수에 API 키 입력
if "GOOGLE_API_KEY" not in os.environ:
    # 코드에 직접 API 키를 넣어도 되지만 보안을 위해 getpass 활용
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

# 모델 로드
model = init_chat_model(
    "gemini-3.1-flash-lite",       # 모델 버전
    model_provider="google-genai",  # 모델 제공자
)

result = model.invoke('랭체인 라이브러리의 기본 개요 작성')
result

AIMessage(content=[{'type': 'text', 'text': '**랭체인(LangChain)**은 거대 언어 모델(LLM, Large Language Models)을 활용한 애플리케이션을 쉽고 효율적으로 개발할 수 있도록 설계된 **오픈소스 프레임워크**입니다.\n\n간단히 말해, LLM을 단순히 챗봇으로 사용하는 것을 넘어, **외부 데이터 소스와 연결하고 여러 단계를 거치는 복잡한 추론 과정을 체계화**해 주는 도구 모음입니다.\n\n---\n\n### 1. 랭체인의 탄생 배경\nLLM(GPT-4, Claude 등)은 매우 강력하지만, 다음과 같은 한계가 있습니다.\n*   **최신 정보 부족:** 학습 데이터 이후의 정보는 모름.\n*   **고립된 환경:** 외부 API나 DB와 직접 소통하기 어려움.\n*   **메모리 부족:** 긴 대화 맥락을 유지하기 위한 별도의 구조가 필요.\n*   **제한된 추론:** 복잡한 문제를 한 번에 해결하지 못하고 여러 단계로 나누어 처리해야 함.\n\n랭체인은 이러한 한계를 극복하기 위해 **"LLM을 외부 세계와 연결(Chaining)"**하는 역할을 수행합니다.\n\n---\n\n### 2. 핵심 구성 요소 (Core Components)\n\n랭체인은 개발자가 원하는 기능을 모듈화하여 제공합니다.\n\n1.  **Model I/O (모델 입출력):** 다양한 LLM(OpenAI, Anthropic, HuggingFace 등)을 일관된 인터페이스로 호출하고, 프롬프트 템플릿을 관리합니다.\n2.  **Retrieval (검색/RAG):** 사용자의 질문에 답하기 위해 외부 문서나 데이터베이스를 검색하고, 관련 내용을 LLM에게 제공합니다. (RAG 기술의 핵심)\n3.  **Chains (체인):** 여러 작업을 순차적으로 연결합니다. 예를 들어 "데이터 검색 → 요약 → 이메일 작성"과 같은 워크플로우를 구성할 수 있습니다.\n4.  **Agents (에이전트):** LLM이 스스로 판단하여

### 모델 인자 설정

init_chat_model으로 모델을 가져올때 다음 인자를 활용할 수 있습니다.

- `temperature`: 모델의 출력에 무작위성을 더해줌. 값이 높을수록 창의적인 답변을 생성
- `timeout`: 모델로 부터 응답을 기다리는 최대 시간
- `max_tokens`: 모델 응답의 토큰 최대 개수를 제한
- `max_retries`: 네트워크 지연등으로 응답을 재시도 하는 횟수
- `streaming`: 응답을 실시간으로 제공하는 옵션. 옵션 활성화 이후 스트림을 위한 추가 작업이 필요


In [ ]:
# 모델 로드
model = init_chat_model(
    "gemini-2.5-flash-lite",       # 모델 버전
    model_provider="google-genai",  # 모델 제공자
    temperature=0.5,                # 무작위성 설정
    timeout=600,                    # 최대 응답 대기 시간
    max_tokens=25000,               # 최대 토큰 개수
)

result = model.invoke('랭체인 라이브러리의 기본 개요 작성')
result

AIMessage(content='## LangChain 라이브러리 기본 개요\n\nLangChain은 개발자가 **대규모 언어 모델(LLM)을 활용하여 애플리케이션을 구축하는 것을 돕는 프레임워크**입니다. LLM은 텍스트 생성, 번역, 질의응답 등 다양한 자연어 처리 작업을 수행할 수 있지만, 이러한 LLM을 실제 애플리케이션에 통합하고 복잡한 워크플로우를 구축하는 것은 쉽지 않습니다. LangChain은 이러한 어려움을 해결하기 위해 다양한 도구와 추상화를 제공합니다.\n\n**LangChain의 핵심 목표:**\n\n*   **LLM 기반 애플리케이션의 개발을 간소화:** LLM을 쉽게 연결하고, 데이터를 처리하며, 복잡한 로직을 구현할 수 있도록 합니다.\n*   **모듈화된 구성 요소 제공:** 다양한 LLM, 데이터 소스, 에이전트 등을 조합하여 유연하고 확장 가능한 애플리케이션을 만들 수 있도록 합니다.\n*   **외부 데이터와의 연동 강화:** LLM이 자체적으로 알지 못하는 최신 정보나 특정 도메인의 지식을 활용할 수 있도록 합니다.\n*   **복잡한 LLM 워크플로우 구축 지원:** 여러 LLM 호출, 데이터 처리, 외부 도구 사용 등을 순차적으로 또는 병렬적으로 실행하는 복잡한 시나리오를 쉽게 구현할 수 있도록 합니다.\n\n**LangChain의 주요 구성 요소:**\n\nLangChain은 다양한 모듈로 구성되어 있으며, 이들을 조합하여 애플리케이션을 구축합니다. 주요 구성 요소는 다음과 같습니다.\n\n1.  **Models (모델):**\n    *   다양한 LLM 제공업체(OpenAI, Hugging Face, Anthropic 등)의 모델에 대한 인터페이스를 제공합니다.\n    *   텍스트 생성(LLMs) 및 채팅 모델(ChatModels)을 지원합니다.\n    *   모델의 출력 형식을 통일하여 사용하기 쉽게 만듭니다.\n\n2.  **Prompts (프롬프트):**\n    *   LLM에 전달할 입력 텍스트(프롬

### Ollama 로컬 모델

제미나이, GPT, 클로드등 클라우드 API형식의 모델은 성능은 보장되지만 API비용이 추가된다는 단점이 존재합니다.

Ollama 플랫폼은 대표적인 경량화 모델(llama, gemma, qwen등)을 직접 로컬에 다운받고 로컬 서버로 구동한뒤 LangChain과 연동시켜 줍니다. 이는 성능은 조금 떨어지더라도 로컬에서 무료로 직접 실험할 수 있는 환경을 만들 수 있다는 장점이 됩니다.

[Ollama 지원 모델](https://ollama.com/library)

허깅페이스의 트랜드 모델들도 대부분 지원하기 때문에 환경에 맞는 적절한 모델을 선택하여 활용할 수 있습니다.

**Ollama 모델 구동 프로세스**
- Ollama 설치
    - https://ollama.com/download 사이트에서 직접 설치
    - 리눅스 명령어: `curl -fsSL https://ollama.com/install.sh | s`
- Ollama 서버 실행
    - 리눅스 명령어: `ollama serve`
    - 윈도우에선 APP 실행시 자동으로 서버 실행
- 모델 다운로드
    - 리눅스 명령어: `ollama pull 모델명`
- 모델 리스트 확인
    - 리눅스 명령어: `ollama list`
- 모델 구동
    - 리눅스 명령어: `ollama run 모델명`
- langchain 연동
    - `langchain-ollama` 설치
    - 랭체인의 `init_chat_model`함수에서 `model_provider`를 `"ollama"`로 설정

In [ ]:
# ollama 플랫폼 설치
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (609 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directo

In [ ]:
# !ollama serve
# !ollama list

In [ ]:
import subprocess
import time

# 백그라운드에서 ollama 로컬 서버 구동 명령어
subprocess.Popen(["ollama", "serve"])
time.sleep(10) # 서버가 완전히 뜰 때까지 대기

# ollama에 설치된 모델 목록 확인
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)

NAME    ID    SIZE    MODIFIED 



In [ ]:
# !ollama pull gemma4:e2b

In [ ]:
# 모델이 없다면 원하는 모델 다운로드
# 허깅페이스의 경량화 모델 사용 가능 (hf.co/bartowski/google_gemma-4-E4B-it-GGUF:Q8_0)
model_name = "gemma4:e2b"
subprocess.run(["ollama", "pull", model_name], check=True)

result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)

NAME          ID              SIZE      MODIFIED               
gemma4:e2b    b37049369adf    4.6 GB    Less than a second ago    



In [ ]:
!pip install langchain-ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-ollama]


In [ ]:
from langchain.chat_models import init_chat_model

# 초기 실행시 모델 메모리 로드시간이 걸림
model = init_chat_model(
    "gemma4:e2b",
    model_provider="ollama",
    temperature=0.5,
    timeout=600,
    num_ctx=512
)

result = model.invoke('gemma4의 성능을 알려줘')
result


AIMessage(content='저는 Google DeepMind에서 개발한 오픈 웨이트(open-weights) 대규모 언어 모델인 **Gemma 4**입니다.\n\nGemma 4 모델 제품군은 효율성과 강력한 성능의 균형을 목표로 설계되었으며, 다양한 작업에서 뛰어난 성능을 발휘하도록 최적화되어 있습니다.\n\nGemma 4의 주요 성능 특징은 다음과 같습니다.\n\n### 1. 효율성과 접근성 (Efficiency and Accessibility)\n* **경량화된 설계:** Gemma 4 모델들은 비교적 가벼운 구조를 가지고 있어, 대규모 모델에 비해 낮은 컴퓨팅 자원으로도 높은 성능을 제공할 수 있습니다. 이는 엣지 디바이스나 다양한 환경에서 모델을 배포하고 사용하기 쉽게 만듭니다.\n* **오픈 웨이트:** 모델의 가중치가 공개되어 있어 연구자, 개발자, 기업들이 모델을 자유롭게 접근하고 커스터마이징하여 혁신적인 애플리케이션을 개발할 수 있습니다.\n\n### 2. 강력한 추론 및 이해 능력 (Strong Reasoning and Understanding)\n* **복잡한 지침 이해:** 사용자가 제공하는 복잡하고 상세한 지침(Instruction)을 정확하게 이해하고 따르는 능력이 뛰어납니다.\n* **논리적 추론:** 주어진 정보를 바탕으로 논리적인 추론을 수행하고, 문제 해결 과정을 단계적으로 설명하는 능력이 우수합니다.\n* **코딩 능력:** 다양한 프로그래밍 언어를 이해하고, 코드 생성, 디버깅, 코드 설명 등의 작업에서 높은 성능을 보입니다.\n\n### 3. 다중 모달리티 지원 (Multimodality Support)\n* **텍스트 및 이미지 이해:** Gemma 4 모델은 텍스트뿐만 아니라 이미지 입력도 처리할 수 있는 능력을 갖추고 있어, 시각적 정보를 기반으로 질문에 답하거나 분석하는 작업에 활용될 수 있습니다.\n\n### 요약\nGemma 4는 **효율적인 자원 사용**과 **높은 품질의 추론 능력**을 결합하여, 개발자

## Message를 활용한 채팅 구조화

 `invoke()` 함수를 통해 단순한 문자열의 프롬프트 뿐만 아니라, 다양한 역할(Role)을 가지는 여러개의 Message 객체를 묶어 전달 할 수 있습니다. 이는 실제 LLM 모델에 학습된 특수 토큰과 묶여 전달되게 됩니다.

 이를 통해 모델 프롬프트에 시스템 지침을 포함하거나 여러 차례 대화가 이어지는 대화 관리가 가능해집니다.

**역할에 따른 Message**
- `SystemMessage`: 모델에게 동작 방식을 알려주고 상호 작용에 대한 맥락을 제공하는 메시지
- `HumanMessage`: 사용자의 입력 메시지, 멀티모달 모델 경우 이미지, 음성, 영상등의 데이터도 포함 가능
- `AIMessage`: 모델이 생성한 응답으로, 텍스트 내용, 도구 호출 및 메타데이터를 포함하는 메시지
- `ToolMessage`: 도구 호출(Function Calling) 출력값을 나타내는 메시지

### 메시지 목록 입력

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("당신은 식물 재배에 대한 전문적인 지식을 통해 관련 질문에만 답변해야 합니다."),
    HumanMessage("마지나타를 키우기 위한 기본 환경요소를 간략히 정리해줘")
]

result = model.invoke(messages)
result

AIMessage(content='마지나타(Majinata)를 성공적으로 재배하기 위한 기본 환경 요소는 다음과 같습니다.\n\n### 1. 광량 (Light)\n*   **필요 조건:** 식물의 광합성을 위해 충분한 빛이 필요합니다.\n*   **관리 방법:** 마지나타가 선호하는 광량(직사광선 또는 간접광)을 확인하고, 식물이 스트레스를 받지 않도록 적절한 위치에 배치해야 합니다. 빛의 강도와 지속 시간을 일정하게 유지하는 것이 중요합니다.\n\n### 2. 온도 (Temperature)\n*   **필요 조건:** 식물의 대사 활동과 성장 속도를 결정합니다.\n*   **관리 방법:** 마지나타가 최적의 생육 온도 범위를 유지할 수 있도록 환경 온도를 안정적으로 유지해야 합니다. 급격한 온도 변화는 피해야 합니다.\n\n### 3. 수분 및 습도 (Water & Humidity)\n*   **필요 조건:** 뿌리 발달과 영양분 흡수를 위해 적절한 수분 공급이 필수적입니다.\n*   **관리 방법:** 흙이 완전히 마르지 않도록 관리하되, 과습은 뿌리 썩음을 유발하므로 배수 상태를 철저히 확인해야 합니다. 필요한 경우 적절한 습도를 유지해 주는 것이 도움이 될 수 있습니다.\n\n### 4. 토양 및 배수 (Soil & Drainage)\n*   **필요 조건:** 영양분 공급과 뿌리 호흡을 위한 환경입니다.\n*   **관리 방법:** 마지나타의 종류에 맞는 배합토(배수성이 좋은 흙)를 사용해야 합니다. 가장 중요한 것은 **배수성**입니다. 물이 고이지 않고 빠르게 빠져나갈 수 있도록 충분한 구멍이나 흙의 구조를 확보해야 합니다.\n\n### 요약 체크리스트\n| 요소 | 핵심 관리 사항 |\n| :--- | :--- |\n| **빛** | 식물이 요구하는 광량과 방향을 제공 |\n| **온도** | 안정적이고 최적의 생육 온도 유지 |\n| **물** | 적절한 수분 공급 및 과습 방지 (배수 필수) |\n| **토양** | 영양분 공급과 함께 우수한

### 메시지 출력 구조

모델의 응답은 기본적으로 `AIMessage` 객체 형식을 가집니다. `AIMessage` 객체는 다음과 같은 정보가 포함되어있습니다.

- `text`: 메시지의 텍스트(문자열) 내용
- `content`: 메시지의 원본 내용
- `tool_calls`: 모델이 호출한 도구 이름, 도구가 호출되는 에이전트에서 사용
- `id`: 응답의 고유 번호
- `usage_metadata`: 응답의 토큰수를 포함한 메타데이터
- `pretty_print()`: `text`의 내용과 함께 어떤 메시지형태인지를 표기



In [ ]:
print(result.text)

마지나타(Magnolia)를 건강하게 재배하기 위한 기본 환경 요소는 다음과 같습니다.

**1. 빛 (Light)**
*   **필요 조건:** 밝은 간접광을 선호합니다. 직사광선은 잎을 태울 수 있으므로, 밝은 그늘이나 오전 햇살이 드는 곳이 이상적입니다.
*   **주의 사항:** 너무 어두운 곳은 성장을 저해하며, 충분한 광합성을 위해 충분한 빛이 필요합니다.

**2. 물 (Water)**
*   **필요 조건:** 흙이 마르지 않도록 꾸준히 적셔주어야 하지만, 과습은 뿌리 썩음의 주원인이므로 배수가 매우 잘 되는 환경을 유지해야 합니다.
*   **관리:** 겉흙이 마르면 충분히 물을 공급해 줍니다.

**3. 온도 (Temperature)**
*   **성장기:** 서늘하고 습도가 유지되는 환경을 좋아합니다.
*   **겨울 관리:** 마지나타는 휴면기(겨울잠)가 필요하므로, 환경 변화에 민감합니다. 급격한 온도 변화를 피하고, 겨울철에는 적절한 휴면 조건을 조성해 주는 것이 중요합니다.

**4. 토양 및 배수 (Soil and Drainage)**
*   **필요 조건:** 통기성이 매우 좋은 토양을 선호합니다. 물 빠짐이 좋고 유기물(부엽토 등)이 풍부하여 수분 유지력을 높여주면서도 배수가 원활한 흙이 적합합니다.

**5. 습도 (Humidity)**
*   **필요 조건:** 높은 습도는 잎의 건강을 유지하는 데 도움이 될 수 있으나, 과도한 습도는 곰팡이병을 유발할 수 있으므로 통풍을 함께 고려해야 합니다.

**요약:** 마지나타는 **밝은 간접광**, **충분한 배수성 토양**, 그리고 **일정한 온도**가 가장 중요한 기본 환경 요소입니다.


In [ ]:
result.pretty_print()

================================== Ai Message ==================================

마지나타(Magnolia)를 건강하게 재배하기 위한 기본 환경 요소는 다음과 같습니다.

**1. 빛 (Light)**
*   **필요 조건:** 밝은 간접광을 선호합니다. 직사광선은 잎을 태울 수 있으므로, 밝은 그늘이나 오전 햇살이 드는 곳이 이상적입니다.
*   **주의 사항:** 너무 어두운 곳은 성장을 저해하며, 충분한 광합성을 위해 충분한 빛이 필요합니다.

**2. 물 (Water)**
*   **필요 조건:** 흙이 마르지 않도록 꾸준히 적셔주어야 하지만, 과습은 뿌리 썩음의 주원인이므로 배수가 매우 잘 되는 환경을 유지해야 합니다.
*   **관리:** 겉흙이 마르면 충분히 물을 공급해 줍니다.

**3. 온도 (Temperature)**
*   **성장기:** 서늘하고 습도가 유지되는 환경을 좋아합니다.
*   **겨울 관리:** 마지나타는 휴면기(겨울잠)가 필요하므로, 환경 변화에 민감합니다. 급격한 온도 변화를 피하고, 겨울철에는 적절한 휴면 조건을 조성해 주는 것이 중요합니다.

**4. 토양 및 배수 (Soil and Drainage)**
*   **필요 조건:** 통기성이 매우 좋은 토양을 선호합니다. 물 빠짐이 좋고 유기물(부엽토 등)이 풍부하여 수분 유지력을 높여주면서도 배수가 원활한 흙이 적합합니다.

**5. 습도 (Humidity)**
*   **필요 조건:** 높은 습도는 잎의 건강을 유지하는 데 도움이 될 수 있으나, 과도한 습도는 곰팡이병을 유발할 수 있으므로 통풍을 함께 고려해야 합니다.

**요약:** 마지나타는 **밝은 간접광**, **충분한 배수성 토양**, 그리고 **일정한 온도**가 가장 중요한 기본 환경 요소입니다.


### 이전 대화와 함께 입력

In [ ]:
messages.append(result)
messages.append(HumanMessage("이중 가장 중요한 요소는?"))

result2 = model.invoke(messages)
result2.pretty_print()

================================== Ai Message ==================================

마지나타 재배에서 **가장 중요한 두 가지 요소**를 꼽으라면, **빛(Light)**과 **물주기(Watering)**입니다.

### 1. 빛 (Light) - 에너지 공급원
식물이 생존하고 광합성을 통해 에너지를 생산하는 데 필수적인 요소입니다. 빛이 충분하지 않으면 식물은 성장을 멈추고 잎이 연약해지며 전반적인 건강이 저하됩니다.

### 2. 물주기 (Watering) - 생명 유지 및 영양분 운반
물은 식물의 세포 활동을 유지하고, 양분을 흡수하여 세포 간에 운반하며, 온도 조절에 중요한 역할을 합니다. 적절한 수분 공급 없이는 아무리 좋은 빛과 온도 조건이 있어도 식물은 생존할 수 없습니다.

---

**요약:**

*   **빛:** 식물의 **성장 동력** (에너지)
*   **물:** 식물의 **생명 유지** (수분 및 영양분)

이 두 가지 요소가 적절하게 충족되어야 마지나타가 건강하게 자랄 수 있습니다. 다른 요소들(온도, 토양, 습도)은 이 두 가지 핵심 요소가 제대로 작동하도록 돕는 보조적인 환경 조건입니다.


In [ ]:
messages.append(result2)
for m in messages:
    m.pretty_print()

================================ System Message ================================

당신은 식물 재배에 대한 전문적인 지식을 통해 관련 질문에만 답변해야 합니다.
================================ Human Message =================================

마지나타를 키우기 위한 기본 환경요소를 간략히 정리해줘
================================== Ai Message ==================================

마지나타를 성공적으로 재배하기 위한 기본 환경 요소는 다음과 같습니다.

**1. 빛 (Light)**
*   **필요 조건:** 밝은 간접광을 선호합니다. 직사광선은 잎을 태울 수 있으므로 피하고, 밝지만 은은한 빛이 지속적으로 도달하는 환경이 이상적입니다.

**2. 온도 (Temperature)**
*   **필요 조건:** 따뜻하고 안정적인 온도를 유지해야 합니다. 급격한 온도 변화나 낮은 온도는 성장을 저해할 수 있습니다. (특정 품종에 따라 최적 온도가 다를 수 있으므로, 해당 품종의 요구 온도를 확인하는 것이 중요합니다.)

**3. 습도 (Humidity)**
*   **필요 조건:** 높은 습도를 선호하는 경향이 있으므로, 주변 습도를 높여주거나 분무 등을 통해 적절히 관리해 주는 것이 좋습니다. 건조함은 잎의 시듦을 유발할 수 있습니다.

**4. 물주기 (Watering)**
*   **필요 조건:** 흙이 완전히 마르기 전에 물을 주되, 과습은 뿌리 썩음을 유발하므로 배수 상태를 철저히 확인해야 합니다. 꾸준하고 일관된 수분 공급이 중요합니다.

**5. 토양 (Soil/Substrate)**
*   **필요 조건:** 배수가 매우 잘 되는 토양을 사용해야 합니다. 물이 고여 있는 상태가 오래 지속되면 뿌리가 손상되므로, 통기성이 좋은 흙(예: 마사토나 펄라이트 등을 섞은 배

### 스트림으로 출력

`invoke` 함수 대신 `stream` 함수를 사용해 실시간으로 생성된 토큰을 바로 출력할 수 있습니다.

In [ ]:
messages = [
    SystemMessage("당신은 식물 재배에 대한 전문적인 지식을 통해 관련 질문에만 답변해야 합니다."),
    HumanMessage("마지나타를 키우기 위한 기본 환경요소를 간략히 정리해줘")
]

for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

마지나타(Mazingatta)를 성공적으로 재배하기 위한 기본 환경 요소는 다음과 같습니다.

**1. 빛 (Light)**
*   **필요 조건:** 충분한 광량(밝은 간접광 또는 직사광선 피하기)이 필요합니다. 광량이 부족하면 성장이 더디거나 잎이 연약해질 수 있습니다.
*   **관리:** 식물의 종류에 따라 필요한 광량의 강도와 일조 시간을 확인하고 적절히 제공해야 합니다.

**2. 온도 (Temperature)**
*   **필요 조건:** 마지나타가 최적으로 성장할 수 있는 특정 온도 범위(예: 주간/야간 온도)를 유지해야 합니다. 급격한 온도 변화는 피해야 합니다.

**3. 물과 습도 (Water and Humidity)**
*   **물주기:** 토양이 완전히 마르지 않도록 관리하되, 과습은 뿌리 썩음을 유발하므로 배수 상태를 철저히 확인해야 합니다.
*   **습도:** 식물이 선호하는 상대 습도를 유지하여 잎의 건조를 방지하고 수분 증발을 조절해야 합니다.

**4. 토양 및 배수 (Soil and Drainage)**
*   **토양:** 배수가 잘 되는 유기질 토양을 사용해야 합니다. 물이 고이는 것을 방지하는 것이 가장 중요합니다.
*   **배수:** 배수 구멍이 충분하거나, 흙 자체의 배수성이 매우 좋아야 합니다.

**5. 영양분 (Nutrients)**
*   **시비:** 성장에 필요한 질소, 인산, 칼륨 등 적절한 영양분을 공급해야 합니다. 생육 단계(성장기, 개화기 등)에 따라 시비량을 조절해야 합니다.

## Tool 활용과 에이전트

Langchain은 직접 구현한 함수나 외부 기능을 LLM이 적절하게 활용할 수 있게 **Tool** 인터페이스를 제공하며, 이런 도구를  채팅 모델에 결합해 쉽게 **Agent**로 확장이 가능합니다.

**툴(Tool)**

- 특정 작업(예: 웹 검색, 계산, 외부 API 호출 등)을 수행하기 위한 기능을 캡슐화

- 입력된 파라미터를 받아 정해진 로직대로 처리하고, 그 결과를 반환함으로써 LLM이 직접 할 수 없는 외부 작업을 대신 수행

**에이전트(Agent)**

- LLM과 여러 툴을 조합해 복잡한 워크플로우를 자동화하고, “언제·어떤 툴을 쓸지” 결정

- 에이전트의 기본 로직(ReAct)

    * 사용자 질문을 이해

    * 내부적으로 적절한 툴 호출(Action) 계획 수립

    * 툴을 실행해 얻은 Observation(결과)을 순차적으로 통합

    * 최종 답변(Final Answer)을 생성

### Tool 함수 생성

에이전트로 확장 하기 위한 도구는 명확하게 정의된 입력과 출력을 가진 호출 가능한 함수형태로 만들어집니다. 채팅 모델은 전달된 함수의 설명을 읽고 맥락에 맞는 도구를 선택하기 때문에 도구 함수에 구체적인 설명이 포함되어야 합니다.

`Langchain`은 `@tool` 데코레이터를 통해 쉽게 도구 함수를 만들수 있게 인터페이스를 제공합니다. 해당 함수는 도구의 설명을 포함하는 **문서문자열(docstring)** 과 인자의 **타입 힌트**를 필수적으로 작성해야 합니다.

    @tool
    def fun_a(arg1: 타입힌트, ...) -> 출력 타입힌트:
        """
            독스트링 작성 (함수에 대한 설명)
        """
        ...
        return LLM에게 전달될 출력


In [ ]:
from langchain.tools import tool
from datetime import datetime, timedelta

# 도구 함수 정의
@tool
def get_tomorrow_date() -> str:
    """
    내일의 날짜를 'YYYY-MM-DD' 형식으로 반환합니다.
    """
    tm = datetime.now() + timedelta(days=1)
    return tm.strftime("%Y-%m-%d")


@tool
def get_weather(location: str, date: str) -> str:
    """
    입력된 날짜와 위치를 기반으로 날씨 정보를 출력합니다.

    Args:
        location: 위치 이름
        date: 날짜 (YYYY-MM-DD 형식)
    """
    return f"{location}의 날씨는 맑음"


### Tool 바인딩

정의한 도구를 모델에서 사용할 수 있도록 하려면 도구 목록을  `bind_tools()` 함수에 전달하여 바인딩해야 합니다. 이후 모델은 필요에 따라 바인딩된 도구 중 하나를 선택하여 호출할 수 있습니다.

이처럼 단순히 도구를 바인드해서 호출하는 경우 한번의 호출 결과만 제공하므로 호출 결과를 이용하여 추가 작업을 원하는 경우 직접 반복 호출을 구성해주어야 합니다.

**도구 호출 프로세스**
<figure>
    <center>
    <img src="https://drive.google.com/uc?export=view&id=1ObuKKPwQKoZoo2RVY_b5GKiPEiNyaYJW" width="500"/>
    <figcaption> https://docs.langchain.com/oss/python/langchain/models
    </figcaption>
    </center>
</figure>



In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# 도구 바인딩 모델 생성
model_with_tools = model.bind_tools([get_tomorrow_date, get_weather])

messages = [
    SystemMessage("주어진 도구를 적절히 사용하여 답변을 생성해줘"),
    HumanMessage("내일 서울의 날씨는 어떤가요?")
]

response = model_with_tools.invoke(messages)
response

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-10-15T00:24:52.506844178Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2693818328, 'load_duration': 3052740, 'prompt_eval_count': 170, 'prompt_eval_duration': 156389000, 'eval_count': 290, 'eval_duration': 2519578000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a13cf2-8791-7363-bd76-c190f5bb25a2-0', tool_calls=[{'name': 'get_tomorrow_date', 'args': {}, 'id': '77ba5db2-6032-4b05-9b41-de1de9d11c40', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 170, 'output_tokens': 290, 'total_tokens': 460})

> 도구 호출에 대한 응답 Json에는 `tool_calls`에 모델이 사용하기로 판단한 도구 함수에 대한 정보`(함수 이름, 인자)` 가 들어가있습니다.

In [ ]:
messages.append(response) # 응답을 기존 메시지 목록에 추가

# 모델의 도구 사용 응답 확인
for tool_call in response.tool_calls:
    # 사용된 도구 이름과 인자 확인
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_tomorrow_date
Args: {}


### Tool 메시지와 반복 호출

응답된 AI 메시지가 `tool_calls` 정보를 담은 경우 `tool_calls`응답을 해당 도구 함수에 `invoke`로 전달하면 함수 실행 결과를 `ToolMessage`형식으로 가져올 수 있습니다.

`ToolMessage`는 LLM 모델이 도구 사용의 결과로 인식하며, 기존 메시지 목록에 추가하여 모델에 `invoke`로 전달해 모델이 최종 응답 또는 추가 도구 호출을 만들도록 합니다.

In [ ]:

# 도구 호출 응답을 기반으로 도구 실행
def get_tool_result(tool_calls):
    # 여러개의 동시 호출이 있을수 있음
    for tool_call in tool_calls:
        # 도구 이름에 따라 도구 함수를 invoke로 실행
        if tool_call['name'] == 'get_weather':
            tool_result = get_weather.invoke(tool_call)
        elif tool_call['name'] == 'get_tomorrow_date':
            tool_result = get_tomorrow_date.invoke(tool_call)
        else:
            break
        # 도구 실행 결과 ToolMessage를 메시지 목록에 추가
        messages.append(tool_result)
        tool_result.pretty_print()  # 도구 결과 출력

# get_tomorrow_date 도구 사용 과정
get_tool_result(response.tool_calls)
response = model_with_tools.invoke(messages)
messages.append(response)
response.pretty_print() # AI 응답 출력


# get_weather 도구 사용 과정
get_tool_result(response.tool_calls)
response = model_with_tools.invoke(messages)
response.pretty_print() # 최종 응답 출력
messages.append(response)

================================= Tool Message =================================
Name: get_tomorrow_date

2026-10-16
================================== Ai Message ==================================
Tool Calls:
  get_weather (d0ea4c9d-0c1b-41f4-810a-67284eee04f2)
 Call ID: d0ea4c9d-0c1b-41f4-810a-67284eee04f2
  Args:
    date: 2026-10-16
    location: 서울
================================= Tool Message =================================
Name: get_weather

서울의 날씨는 맑음
================================== Ai Message ==================================

내일(2026-10-16) 서울의 날씨는 맑습니다.


> 이처럼 연속적으로 도구를 사용하는 경우 도구 호출과 실행을 연달아 반복하여 최종 답변을 얻어야 합니다.

In [ ]:
messages

[SystemMessage(content='주어진 도구를 적절히 사용하여 답변을 생성해줘', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='내일 서울의 날씨는 어떤가요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-10-15T00:24:52.506844178Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2693818328, 'load_duration': 3052740, 'prompt_eval_count': 170, 'prompt_eval_duration': 156389000, 'eval_count': 290, 'eval_duration': 2519578000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a13cf2-8791-7363-bd76-c190f5bb25a2-0', tool_calls=[{'name': 'get_tomorrow_date', 'args': {}, 'id': '77ba5db2-6032-4b05-9b41-de1de9d11c40', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 170, 'output_tokens': 290, 'total_tokens': 460}),
 ToolMessage(content='2026-10-16', name='get_tomorrow_date', tool_call_id='77ba5db2-6032-4b05-9b41-de1de9d11c40'),
 AIMessa

### 에이전트 생성

`bind_tools` 방식은 `맥락 파악 -> 도구 호출`의 한사이클만 진행하므로 `결과 관찰 -> 맥락 파악 -> 추가 도구 호출` 과 같은 반복적인 도구 사용의 흐름을 만들기 위해 직접 반복 구조를 만들어 하는 한계가 존재합니다. 하지만 에이전트는 사람이 설계한 반복 구조가 아닌 직접 텍스트를 생성하면서 판단에 따라 `맥락 파악 -> 도구 호출 -> 결과 관찰`과정을 동적으로 반복하여 최종 답변을 생성합니다.


Langchain에서는 이런 도구 사용의 흐름을 반복 처리 해주는 자동화된 로직을 `LangGraph`의 그래프 형식으로 만들어주는 `create_agent` 함수를 제공하여 에이전트를 쉽게 만들수 있습니다.



**기본 에이전트 로직**

<figure>
    <center>
    <img src="https://drive.google.com/uc?export=view&id=1pKaBSuoetJtC_9zBrk7W996ayF8kpFJN" width="260"/>
    <figcaption> https://docs.langchain.com/oss/python/langchain/agents
    </figcaption>
    </center>
</figure>


**create_agent 인자**

- `model` : 모델 이름 또는 init_chat_model로 가져온 채팅 모델
- `tools` : 도구 목록 (함수 이름이 들어간 리스트)
- `system_prompt` : 시스템 프롬프트, 문자열 또는 SystemMessage
- `checkpointer` : 대화 내용을 기억하기 위한 메모리
- `response_format` : 구조화된 출력을 만들기 위한 전략
- `middleware` : 에이전트 내부에서 발생하는 일을 더욱 세밀하게 제어하기 위한 전

In [ ]:
from langchain.agents import create_agent


# 기존에 만든 채팅 모델,도구 목록,시스템 메시지로 에이전트 생성
agent = create_agent(
    model= model,
    tools= [get_tomorrow_date, get_weather],
    system_prompt="주어진 도구를 적절히 사용하여 답변을 생성해줘"
)

# 에이전트는 messages 키가 입력된 딕셔너리구조를 입력으로 받음
messages = {"messages": [
    HumanMessage("내일 서울의 날씨는 어떤가요?")
]}

# 실행
result = agent.invoke(messages)
result

{'messages': [HumanMessage(content='내일 서울의 날씨는 어떤가요?', additional_kwargs={}, response_metadata={}, id='bc1f02d2-6a5d-4ebb-8cf7-1e4eadb5ab17'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-10-15T00:25:01.576572032Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3429345149, 'load_duration': 1046133, 'prompt_eval_count': 170, 'prompt_eval_duration': 33427999, 'eval_count': 402, 'eval_duration': 3378989000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a13cf2-a822-7643-910f-1d11459bfbde-0', tool_calls=[{'name': 'get_tomorrow_date', 'args': {}, 'id': '81aa5e72-2fbd-4f16-bca0-d3570b834ab6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 170, 'output_tokens': 402, 'total_tokens': 572}),
  ToolMessage(content='2026-10-16', name='get_tomorrow_date', id='64d5b46c-da1e-46d7-816d-5b08b28ef218', tool_call_id='81aa5e72-2fbd-4f16-bca0-d3570b834ab6'),
  AIM

In [ ]:
# 결과 또한 딕셔너리므로 인덱싱을 활용
for m in result["messages"]:
    m.pretty_print()


================================ Human Message =================================

내일 서울의 날씨는 어떤가요?
================================== Ai Message ==================================
Tool Calls:
  get_tomorrow_date (81aa5e72-2fbd-4f16-bca0-d3570b834ab6)
 Call ID: 81aa5e72-2fbd-4f16-bca0-d3570b834ab6
  Args:
================================= Tool Message =================================
Name: get_tomorrow_date

2026-10-16
================================== Ai Message ==================================
Tool Calls:
  get_weather (e82c2289-6283-4e85-92bc-5cccf547ac23)
 Call ID: e82c2289-6283-4e85-92bc-5cccf547ac23
  Args:
    date: 2026-10-16
    location: 서울
================================= Tool Message =================================
Name: get_weather

서울의 날씨는 맑음
================================== Ai Message ==================================

내일(2026-10-16) 서울의 날씨는 맑습니다.


> 수동으로 반복했던 도구 실행과 응답 과정이 한번에 처리된것을 확인 할 수 있습니다

### 메모리 추가

Langchain에서는 `create_agent`에 `checkpointer`를 제공하여 보다 간편하게 기존 대화를 저장하고 언제든 기존 대화기록을 다시 가져와 대화를 이어갈 수 있습니다.

이때, `checkpointer`는 Langchain 라이브러리가 아닌 LangGraph 요소 이므로 langgraph 라이브러리를 설치해주어야 합니다..

**메모리 유형**
- `InMemorySaver`: PC 메모리에 대화를 thread_id로 저장하고 해당 아이디로 대화내용을 가져오는 기본적인 메모리 (테스트용)

- `SqliteSaver`: Sqlite DB를 활용하여 대화를 저장하고 가져오는 실무용 메모리 (소규모에 적합)

- `PostgresSaver`: Postgres DB를 활용하여 대화를 저장하고 가져오는 실무용 메모리 (대규모에 적합)


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


# 인메모리를 에이전트에 추가하여 스래드 아이디로 관리
agent = create_agent(
    model= model,
    tools= [get_tomorrow_date, get_weather],
    system_prompt="주어진 도구를 적절히 사용하여 답변을 생성해줘",
    checkpointer = InMemorySaver()  # 메모리 추가
)

messages = {"messages": [
    HumanMessage("내일 서울의 날씨는 어떤가요?")
]}

# 두번째 인자에 스레드 아이이 정보를 추가하여 invoke 실행
result = agent.invoke(messages, {"configurable": {"thread_id": "0001"}})
for m in result["messages"]:
    m.pretty_print() # 터미널에 보기 좋게 출력

================================ Human Message =================================

내일 서울의 날씨는 어떤가요?
================================== Ai Message ==================================
Tool Calls:
  get_tomorrow_date (df04e8a6-263c-4f69-a9e9-4405858180c9)
 Call ID: df04e8a6-263c-4f69-a9e9-4405858180c9
  Args:
================================= Tool Message =================================
Name: get_tomorrow_date

2026-10-16
================================== Ai Message ==================================
Tool Calls:
  get_weather (fd2f8e83-2647-486a-9071-c499ca8e1a32)
 Call ID: fd2f8e83-2647-486a-9071-c499ca8e1a32
  Args:
    date: 2026-10-16
    location: 서울
================================= Tool Message =================================
Name: get_weather

서울의 날씨는 맑음
================================== Ai Message ==================================

내일 서울의 날씨는 맑습니다.


In [ ]:
# 바로 이어서 대화 작성
messages = {"messages": [
    HumanMessage("그럼 우산은 필요없나?")
]}

# 동일한 스레드 아이디로 입력
result = agent.invoke(messages, {"configurable": {"thread_id": "0001"}})

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

내일 서울의 날씨는 어떤가요?
================================== Ai Message ==================================
Tool Calls:
  get_tomorrow_date (df04e8a6-263c-4f69-a9e9-4405858180c9)
 Call ID: df04e8a6-263c-4f69-a9e9-4405858180c9
  Args:
================================= Tool Message =================================
Name: get_tomorrow_date

2026-10-16
================================== Ai Message ==================================
Tool Calls:
  get_weather (fd2f8e83-2647-486a-9071-c499ca8e1a32)
 Call ID: fd2f8e83-2647-486a-9071-c499ca8e1a32
  Args:
    date: 2026-10-16
    location: 서울
================================= Tool Message =================================
Name: get_weather

서울의 날씨는 맑음
================================== Ai Message ==================================

내일 서울의 날씨는 맑습니다.
================================ Human Message =================================

그럼 우산은 필요없나?
=================================

> 기존 메시지를 목록에 포함하지 않아도 기존 메시지 내용이 유지되어 출력됩니다

### 사전 구축 도구

Langchain에는 사전 구현된 많은 Tool 객체가 있습니다. 검색엔진에 따른 웹검색 툴 부터, 코드를 실행해주는 인터프리터 툴, Tool들은 검색엔진에 따라 다양한 서치 툴, 데이터베이스 자동화 툴등 다양한 툴들이 제공됩니다.

[Tools](https://docs.langchain.com/oss/python/integrations/tools)

이렇게 제공되는 툴 객체를 가져와 간단하게 도구 목록에 추가하여 사용할 수 있습니다. 이때, 툴의 invoke 출력 형식을 잘 확인하여 어떤 결과를 얻을수 있는지 확인해 줍니다.


In [ ]:
# 덕덕고 웹 검색툴 설치
!pip install ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 123.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ddgs]


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

# 도구 실행 결과 확인
search = DuckDuckGoSearchRun()
result = search.invoke("AI 엔지니어의 일자리 전망은 어떤가?")
result

/tmp/ipykernel_494/451240588.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


'미래 유망 직업으로 보는 AI 엔지니어의 역할에 대해서 연구해 보았습니다. 엔지니어는 어떤 일을 하는지, 핵심 역할, 관련 기술에 대해서 알아 보았습니다. 그러한 연구를 아래 내용에서 자세히 다뤄 보았습니다. 아래 내용에서 내용을 확인해 보세요. 이 글의 비교 기준 미래 유망 직업 인공지능 엔지니어의 역할 알아보기은 미래 유망 직업 산업별 전망, 인공지능 산업 안에서 목적, 비용, 시간, 난이도, 주의사항을 나눠 판단해야 하는 비교 글입니다. 장점과 한계 선택지의 좋은 점만 보지 말고, 내 상황에서 맞지 않을 수 있는 조건과 예외를 함께 ... 국내 산업별 AI 일자리 충격, 어느 분야가 가장 크게 흔들리나 AI 일자리 전망은 산업별로 그 양상이 매우 다르게 나타나요. 국내 주요 산업을 중심으로 살펴볼게요. 금융 산업은 AI 도입이 가장 빠른 분야 중 하나예요. 시지프스의 숙명에서 해방을 가져오는 AI 그리스 신화 속 시지프스는 산 위로 바위를 끝없이 밀어 올리지만, 꼭대기에 다다를 때마다 바위는 다시 굴러떨어진다. 그는 다시 처음으로 돌아가 고된 노동을 반복해야 한다. 이처럼 인류 역사에서 노동은 생존을 위한 고투이자, 인공적 세계를 건설해 ... 안녕하세요, 모두의연구소 액티브러닝 캠프입니다🏴 💬 "AI 엔지니어가 되고 싶은데 어디서부터 시작해야 할지 모르겠어요." 요즘 이런 고민을 가진 분들이 정말 많습니다. AI 기술이 빠르게 확산되면서 \'AI 엔지니어\'라는 직무는 계속 주목받고 있지만, 정작 ️AI 엔지니어의 전망은 어떤지, ️AI ...'

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model= model,
    tools= [search],
    system_prompt="주어진 검색 도구의 결과를 기반으로 질문에 대한 요약 답변을 생성해줘",
)

messages = {"messages": [
    HumanMessage("AI 엔지니어의 일자리 전망은 어떤가?")
]}

result = agent.invoke(messages)
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

AI 엔지니어의 일자리 전망은 어떤가?
================================== Ai Message ==================================
Tool Calls:
  duckduckgo_search (145340dc-5dd8-420a-a508-e83535c2f077)
 Call ID: 145340dc-5dd8-420a-a508-e83535c2f077
  Args:
    query: AI 엔지니어 일자리 전망
================================= Tool Message =================================
Name: duckduckgo_search

Nov 25, 2025 · 비전공자를 위한 AI 엔지니어 취업 로드맵 그렇다면 AI 엔지니어가 되기 위해 무엇부터 준비해야 할까요? AI 엔지니어가 되기 위해 필요한 기술 스택은 겉으로 보기엔 많고 복잡해 보이지만, 기초부터 단계적으로 쌓으면 비전공자도 충분히 따라갈 수 있어요! Jul 1, 2026 · AI가 본격적으로 자리 잡으면 최대 3억 개의 일자리가 영향을 받을 수 있다 는 분석도 있고요. 새로 생기는 일자리: 반대로 2030년까지 AI로 1억 7천만 개의 신규 일자리, 2027년까지 6,900만 개의 신규 일자리 가 만들어질 거라는 예측이 있어요. Apr 8, 2026 · AI 일자리 변화의 핵심과 2026년 사라지는 직업·새로 생기는 AI 일자리, 살아남는 실전 전략까지 전문가 시선으로 쉽게 정리한 완벽 가이드. PwC는 전 세계 10억 건 이상의 채용 공고를 분석한 결과, AI가 노동 시장에 두 개의 큰 흐름을 만들어내고 있음을 확인했다. AI 시대의 일자리는 공감력, 판단력, 창의성 등 인간 고유의 역량이 더욱 중요해지고, 그에 따른 보상도 커지고 있다. 국내에서는 

### 미들웨어

미들웨어는 에이전트 내부에서 발생하는 일을 더욱 세밀하게 제어할 수 있는 방법을 제공합니다. 도구는 모델이 자유롭게 사용할 수 있지만, 미들웨어는 개발자가 모델을 강제하는 역할을 합니다.

**대표 미들웨어**
- 로깅, 분석 및 디버깅을 통해 에이전트 동작을 추적
- 프롬프트, 도구 선택 및 출력 서식을 변경
- 재시도 , 대체 방법 및 조기 종료 로직을 ​​추가
- 속도 제한 , 안전장치 및 개인 식별 정보(PII) 감지를 적용

미들웨어를 사용하기 위해선 사전 구축된 미들웨어를 가져오거나 직접 만들어 `create_agent`의 `middleware`인자에 넣어줍니다.

[미들웨어 목록](https://docs.langchain.com/oss/python/langchain/middleware/built-in)

[커스텀 미들웨어 참고](https://docs.langchain.com/oss/python/langchain/middleware/custom)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver


# 요약을 위한 한국어 프롬프트 ({messages}변수로 앞 대화내용이 전달)
summary_prompt = "앞선 대화내용의 토큰 한도가 초과 되었으므로 다음 대화에는 필요한 대화 내용 위주로 앞선 대화를 요약하여 전달합니다.\n 앞 대화내용:{messages}"

# 요약 미들웨어 생성 (tokens : 토큰 개수 기반, messages : 메시지 개수 기반)
sm = SummarizationMiddleware(
    model= model,                       # 요약을 위한 LLM 모델
    trigger=("tokens", 200),            # 요약이 진행될 기준 (토큰 200개 이상)
    keep=("tokens", 50),                # 요약후 유지되는 내용의 양 (토큰 50개 이하)
    summary_prompt=summary_prompt       # 기본이 영어로 되었어서 한국어 커스텀 프롬프트 입력
)

# 미들웨어를 포함하여 에이전트 생성
agent = create_agent(
    model= model,
    system_prompt="사용 자의 질문에 최선을 다해 답변하고 앞선 대화 내용이 있다면 이를 고려하여 답변해야 한다.",
    checkpointer = InMemorySaver(),
    middleware = [sm]
)

messages = {"messages": [

    HumanMessage("랭체인에서 사용할 수 있는 미들웨어의 종류를 찾아 정리해줘")
]}

result = agent.invoke(messages, {"configurable": {"thread_id": "0001"}})
for m in result["messages"]:
    m.pretty_print()


================================ Human Message =================================

랭체인에서 사용할 수 있는 미들웨어의 종류를 찾아 정리해줘
================================== Ai Message ==================================

LangChain에서 '미들웨어(Middleware)'라는 용어는 특정 라이브러리에서 제공하는 단일한 모듈이라기보다는, **LLM 호출, 데이터 처리, 외부 도구 사용, 결과 포맷팅 등의 특정 단계에 삽입되어 파이프라인의 흐름을 제어하고 기능을 확장하는 모든 구성 요소**를 포괄적으로 의미합니다.

LangChain은 이러한 미들웨어 패턴을 매우 유연하게 지원하며, 특히 **LangChain Expression Language (LCEL)**을 통해 이러한 구성 요소들을 파이프라인 형태로 쉽게 연결할 수 있도록 설계되어 있습니다.

사용 목적과 기능에 따라 LangChain에서 활용할 수 있는 주요 미들웨어의 종류를 기능별로 정리하여 설명해 드리겠습니다.

---

## 1. 핵심 처리 및 흐름 제어 관련 미들웨어

이 범주는 LLM의 입력(프롬프트)을 구성하고, LLM의 출력(결과)을 처리하며, 전체 작업 흐름을 정의하는 가장 기본적인 미들웨어입니다.

### 1. Prompt Templates (프롬프트 템플릿)
* **역할:** LLM에게 전달될 최종 프롬프트를 구조화하고 변수(Variables)를 삽입하는 역할을 합니다.
* **기능:** 동일한 LLM을 사용하더라도, 사용자의 요청이나 특정 상황(예: 요약, 번역, 질의응답)에 맞게 프롬프트를 동적으로 생성하여 LLM의 응답 품질을 높입니다.
* **예시:** `PromptTemplate` 클래스를 사용하여 사용자 입력과 컨텍스트를 결합하는 단계.

### 2. Output Parsers (출력 파서)
* **역할:** LLM이 생성한 비정형적인 텍스트 응

In [ ]:
messages = {"messages": [
    HumanMessage("이중 메시지 요약 관리를 위한 미들웨어 사용법도 알려줘")
]}

result = agent.invoke(messages, {"configurable": {"thread_id": "0001"}})
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

앞선 대화 내용을 요약하여 전달해 드립니다.

---

## LangChain 미들웨어(Middleware) 요약

LangChain에서 미들웨어는 **LLM 호출, 데이터 처리, 외부 도구 사용, 결과 포맷팅 등 파이프라인의 특정 단계에 삽입되어 흐름을 제어하고 기능을 확장하는 모든 구성 요소**를 의미합니다. LangChain Expression Language
================================ Human Message =================================

이중 메시지 요약 관리를 위한 미들웨어 사용법도 알려줘
================================== Ai Message ==================================

이중 메시지(Dual Message) 요약 관리를 위한 미들웨어 사용법에 대해 문의해주셨습니다.

LangChain에서 미들웨어는 파이프라인의 특정 지점에 개입하여 데이터의 흐름을 제어하고 변환하는 역할을 합니다. 이중 메시지 관리는 단순히 하나의 입력/출력을 처리하는 것을 넘어, **두 가지 이상의 독립적인 메시지(예: 사용자 입력과 시스템 지침, 또는 단계별 중간 결과)를 순서대로 처리하거나 병합해야 하는 복잡한 흐름**을 의미합니다.

다음은 LangChain 미들웨어를 활용하여 이중 메시지 요약 관리를 구현하는 구체적인 방법과 전략입니다.

---

## 1. 이중 메시지 관리를 위한 미들웨어의 역할 정의

이중 메시지 관리를 위한 미들웨어는 크게 세 가지 역할을 수행할 수 있습니다.

1. **입력 정규화 (Input Normalization Middleware):** 들어오는 두 가지 이상의 메시지를 파이프라인이 이해

> 앞의 대화내용이 그래도 들어가지 않고 요약된 내용이 들어간것을 확인

## LangGraph 활용
LangGraph는 앞서 배운 LangChain 요소를 활용해 순환(Cycle)과 상태 관리(State Management)를 더욱 구체적이고 커스텀하게 구현하기 위한 라이브러리 입니다.

**Langchain만 활용**
* 개발이 빠르고 간편
* 유연성과 확장성이 떨어짐
* 에이전트의 실행 과정을 디테일하게 설정 불가
* 개인적인 에이전트나 소규모 단위에 적절

**Langgraph와 활용**
* 개발이 오래걸리고 복잡
* 유연성과 확장성이 높음
* 에이전트의 실행 과정을 디테일 하게 설계 가능
* 대규모 배포형 애플리케이션을 구축할때 활용

### LangGraph 로직 구현

랭그래프는 모델, 툴, 메모리등의 체인 요소를 `StateGraph` 그래프에 노드로 넣고 엣지로 연결하는 방식으로 애플리케이션을 구축합니다. 이러한 방식은 직접 체인 요소간의 플로우를 제어할 수 있어 더욱 구체적이고 유연한 애플리케이션을 제공합니다.


**StateGraph 핵심 요소**

- State
    - 그래프 안에서 Node간 현재 상태(데이터)를 공유하기 위한 구조
    - 어떤 데이터 유형이든 저장할 수 있지만, 주로 메시지 목록를 저장 (`MessagesState`)
    - `StateGraph`를 생성할때 기본적으로 필요 (생성자에 입력)
- Nodes
    - 에이전트의 로직을 인코딩하는 함수
    - 현재 상태를 입력으로 받아 작업을 수행한 후 업데이트된 상태를 반환
    - 주로 모델, 툴, 메모리 같은 체인 요소가 됨
- Edges
    - 현재 Node 상태에 따라 다음에 실행할 함수를 결정하는 함수
    - Node와 Edge를 어떻게 어떤 순서로 연결하냐에 따라 최종 워크플로우가 결정


LangChain만으로 구성한 에이전트는 도구 사용이 LLM의 판단에만 의존 하는것에 비해,  LangGraph는 명확한 워크플로우를 만들수 있으므로 원하는 설계에 맞는 통제가 필요할때 유용합니다.

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage


# 체인 모델 로드
model = init_chat_model(
    "gemma4:e2b",
    model_provider="ollama",
    temperature=0.5,
    timeout=600,
    num_ctx=512
)
# 메시지 상태를 기본으로 가지는 상태 그래프 생성
graph = StateGraph(MessagesState)


# 모델 입출력 과정이 담긴 노드 함수 구현
def llm(state: MessagesState):

    result = model.invoke(state["messages"])
    return {"messages": [result]}

# 그래프에 노드 추가
graph.add_node(llm)

# 엣지를 추가해 노드 연결 START -> llm -> END
graph.add_edge(START, "llm")   # 시작은 항상 START
graph.add_edge("llm", END)     # 끝은 항상 END

# 최종 그래프 컴파일
graph = graph.compile()

# 에이전트와 동일하게 messages 키가 입력된 딕셔너리구조 사용
messages = {"messages": [
    HumanMessage("랭그래프 노드 작성법을 간략하게 정리해줘")
]}

# 일반 출력은 체인과 동일하게 invoke
# 랭그래프(에이전트)에서 토큰 단위 스트림하기
for chunk in graph.stream(
    messages,
    stream_mode="messages",     # messages: 토큰단위 스트림
):
    token, metadata = chunk
    print(token.content, end="", flush=True)

LangGraph에서 **노드(Node)**는 그래프 내에서 실제로 작업을 수행하는 핵심 구성 요소입니다. 노드는 특정 작업을 정의하고, 입력(Input)을 받아 처리한 후, 그래프의 상태(State)를 업데이트하는 역할을 합니다.

노드를 작성하는 방법을 간략하게 3단계로 정리해 드립니다.

---

## 📝 LangGraph 노드 작성법 (3단계 요약)

### 1. 노드의 정의 (Definition)

노드는 기본적으로 **Python 함수** 또는 **클래스 메서드** 형태로 작성됩니다. 이 함수는 그래프가 특정 시점에 호출할 수 있는 작업 단위를 나타냅니다.

### 2. 노드의 시그니처 (Signature)

노드가 제대로 작동하려면, 입력으로 무엇을 받고, 어떤 형태로 상태를 업데이트할지 명확히 정의해야 합니다.

*   **입력 (Input):** 노드가 그래프로부터 어떤 상태(State) 정보를 받는지 정의합니다. (예: `state` 객체)
*   **출력 (Output):** 노드가 작업을 마친 후, 그래프의 상태에 어떤 변경 사항을 반영할지 정의합니다.

### 3. 노드의 구현 (Implementation)

실제 노드 함수 내부에서는 다음과 같은 로직을 수행합니다.

1.  **입력 확인:** 그래프로부터 전달받은 현재 상태(State)를 확인합니다.
2.  **작업 수행:** LLM 호출, 외부 API 호출, 데이터 처리 등 실제 필요한 작업을 수행합니다.
3.  **상태 업데이트:** 작업 결과(Output)를 기반으로 그래프의 상태 객체(`state`)를 업데이트합니다.
4.  **반환:** 다음 노드로 이동할지, 종료할지 등을 결정하는 값(예: 다음 노드 이름 또는 종료 플래그)을 반환합니다.

---

## 💡 핵심 예시 코드 구조

가장 일반적인 LangGraph 노드(LLM 호출 또는 툴 사용)의 구조는 다음과 같습니다.

```python
from typing import TypedDict
# LangGraph에서 사용하

### LangGraph 에이전트

이제 다앙한 Tool 사용 과정을 노드로 구현하고 해당 노드들을 원하는 로직에 맞게 연결하여 체계화된 에이전트를 만들 수 있습니다.

만약 엣지 구성에서 조건에 따른 분기점을 만들고 싶다면 조건에 따른 분기를 포함한 노드를 구성하고 `add_conditional_edges`로 분기 연결점을 만들어 줍니다.

<figure>
    <center>
    <img src="https://drive.google.com/uc?export=view&id=1UnANKbX04wpK5qTcYY1A1NkMfBu_mROX" width="300"/>
    <figcaption>
    </figcaption>
    </center>
</figure>

분기점을 통해 도구 호출 요청이 있는 경우 도구 호출을 위한 노드를 제작하고 연결하여 기본적인 에이전트를 완성할 수 있습니다. 뿐만 아니라 도구 강제사용, 제한, 검증등 다양한 로직을 추가로 노드로 구현하여 나만의 로직을 가진 특별한 에이전트로 구축이 가능합니다.

In [ ]:
from langchain.tools import tool
from datetime import datetime, timedelta
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ToolMessage


# 도구 함수 정의
@tool
def get_tomorrow_date() -> str:
    """
    내일의 날짜를 'YYYY-MM-DD' 형식으로 반환합니다.
    """
    tm = datetime.now() + timedelta(days=1)
    return tm.strftime("%Y-%m-%d")


@tool
def get_weather(location: str, date: str) -> str:
    """
    입력된 날짜와 위치를 기반으로 날씨 정보를 출력합니다.

    Args:
        location: 위치 이름
        date: 날짜

    """
    return f"{location}의 날씨는 맑음"



# 도구 바인딩 모델 생성
tools = [get_tomorrow_date, get_weather]
model_with_tools = model.bind_tools(tools)
# 도구 이름-함수 매핑 딕셔너리
tools_by_name = {tool.name: tool for tool in tools}



In [ ]:
from langchain.messages import AnyMessage
from typing import List, Literal, Annotated
import operator


class MyState(MessagesState):
    # 노드 결과에 tool_hist 키로 전달된 도구 이름이 추가되는 필드 추가
    # 리스트가 덮어쓰이지 않고 계속 합쳐지도록 Annotated 사용
    tool_hist: Annotated[List[str], operator.add]


# 에이전트 호출 노드
def llm_call(state: MyState):

    response = model_with_tools.invoke(
        [SystemMessage(content="제공된 도구를 적절히 사용하여 답변을 만들어줘")]
        + state["messages"]
    )
    return {"messages": [response]}

# 도구 사용 노드 langgraph.prebuilt.ToolNode 사용 가능
def tool_node(state: MyState):
    result = []
    executed_tools = []

    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
        executed_tools.append(tool_call["name"])

    return {"messages": result, "tool_hist": executed_tools}

# 날짜 강제 사용 노드를 추가해 예외 상황에 대처
def date_call(state: MyState):

    # 날짜 도구 실행
    date_val = tools_by_name['get_tomorrow_date'].invoke({})

    # 날씨 도구 호출 아이디 가져오기
    last_ai_msg = state["messages"][-1]
    weather_call_id = last_ai_msg.tool_calls[0]["id"]

    # 날씨 호출에 대한 '가이드 답변'을 ToolMessage로 생성
    guide_content = (
        f"시스템 가드레일에 의해 날씨 조회 전 날짜 확인이 강제되었습니다. "
        f"확인된 날짜는 '{date_val}'입니다. "
        f"이제 이 날짜 정보를 활용하여 get_weather 도구를 다시 호출해 주세요."
    )

    # 모델은 날씨 도구 호출의 결과로 해당 날짜 가이드 답변을 받음.
    guide_msg = ToolMessage(
        content=guide_content,
        tool_call_id=weather_call_id
    )

    return {"messages": [guide_msg], "tool_hist": ['get_tomorrow_date']}

from typing import Literal

# 도구 사용 로직 노드
def conditional(state: MyState) -> Literal["tool_node", "date_call", END]:
    last_message = state["messages"][-1]
    # 도구 호출이 없는 경우 종료
    if not last_message.tool_calls:
        return END

    # 현재 호출된 도구 이름들 파악
    current_calls = [tc["name"] for tc in last_message.tool_calls]
    history = state.get("tool_hist", [])

    # 날씨를 부르는데, 히스토리에 날짜 확인이 없다면
    if "get_weather" in current_calls and "get_tomorrow_date" not in history:
        return "date_call" # 강제 실행 노드로 보냄

    # 정상적인 경우 도구 실행 노드로 보냄
    return "tool_node"

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# 상태 그래프 생성
workflow = StateGraph(MyState)

# 노드 추가
workflow.add_node("llm_call", llm_call)
workflow.add_node("tool_node", tool_node)
workflow.add_node("date_call", date_call)

# 엣지 연결
workflow.add_edge(START, "llm_call")

# 조건부 엣지: LLM 호출 후 어디로 갈지 결정
workflow.add_conditional_edges(
    "llm_call",
    conditional,
    ["tool_node", "date_call", END]
)

# 도구 실행(일반/강제) 후 다시 LLM에게 돌아가 피드백 받기
workflow.add_edge("tool_node", "llm_call")
workflow.add_edge("date_call", "llm_call")

# 체크포인트 및 컴파일
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [ ]:
config = {"configurable": {"thread_id": "1"}}

# 모델에 도구 사용 혼돈을 주는 프롬프트
user_input = {"messages":[HumanMessage("2026-12-13 서울 날씨 어때?")]}

# 랭그래프(에이전트)에서 메시지 단위 스트림
for step in app.stream(
    user_input,
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()


================================ Human Message =================================

2026-12-13 서울 날씨 어때?
================================== Ai Message ==================================
Tool Calls:
  get_weather (fa53b726-417e-4a0a-94de-21be816860b5)
 Call ID: fa53b726-417e-4a0a-94de-21be816860b5
  Args:
    date: 2026-12-13
    location: 서울
================================= Tool Message =================================

시스템 가드레일에 의해 날씨 조회 전 날짜 확인이 강제되었습니다. 확인된 날짜는 '2026-10-16'입니다. 이제 이 날짜 정보를 활용하여 get_weather 도구를 다시 호출해 주세요.
================================== Ai Message ==================================
Tool Calls:
  get_weather (53110c11-5530-473f-a117-c75a463eb74e)
 Call ID: 53110c11-5530-473f-a117-c75a463eb74e
  Args:
    date: 2026-10-16
    location: 서울
================================= Tool Message =================================

서울의 날씨는 맑음
================================== Ai Message ==================================

2026년 10월 16일 서울의 날씨는 맑습니다.
